# 06 – RQ4: feature matching on HPatches
In this notebook, we compare SIFT, ORB and our small learned descriptor
(TinyDescNet) on the HPatches benchmark. We measure matching accuracy under
viewpoint and illumination changes on the image sequences, and matching
accuracy on the Easy, Hard and Tough patch splits. The same protocol runs on
our own aerial frames in `scripts/matching_experiments.py`.


In [ ]:
# Install the packages we need.
!pip -q install ultralytics rtmlib onnxruntime-gpu
import torch
print('CUDA available:', torch.cuda.is_available())

# Load the project code.
from pathlib import Path
SRC_ZIP = None
if SRC_ZIP is None:
    from google.colab import files
    up = files.upload()
    SRC_ZIP = next(iter(up))
!mkdir -p /content/project && unzip -q -o "$SRC_ZIP" -d /content/project
import sys
sys.path.insert(0, '/content/project/src')
sys.path.insert(0, '/content/project')

# Results are saved to Google Drive so they survive a disconnect.
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/sar_project_results'); OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Download Okutama-Action files from the public Dropbox folder.
OKUTAMA_BASE = ('https://www.dropbox.com/scl/fo/9qvpsb3fsamvqzsa12149/'
                'APTyV-f01XLnJ0WFpZSBLOE?preview={name}&rlkey=7u7131amaul29amyr4jbnnu03&dl=1')

def fetch_okutama(name, dest='/content/data/okutama'):
    """Download and unpack one Okutama archive unless it is already present."""
    import subprocess, pathlib
    d = pathlib.Path(dest); d.mkdir(parents=True, exist_ok=True)
    zp = d / name
    if not zp.exists():
        subprocess.run(['curl', '-L', '-o', str(zp), OKUTAMA_BASE.format(name=name)], check=True)
    subprocess.run(['unzip', '-q', '-o', str(zp), '-d', str(d)], check=True)
    return d


In [ ]:
# Download the HPatches sequences: 116 sequences of 6 images each, with
# ground-truth homographies (about 4.2 GB).
%cd /content
!curl -sL -o hp.zip https://huggingface.co/datasets/vbalnt/hpatches/resolve/main/hpatches-sequences-release.zip
!mkdir -p data/raw && unzip -q -o hp.zip -d data/raw && rm hp.zip
import config
from pathlib import Path
config.HPATCHES_DIR = Path('/content/data/raw/hpatches-sequences-release')


In [ ]:
# Train TinyDescNet on aerial patches from the Okutama sample frames, with
# the same self-supervised procedure as the local script and a larger budget.
fetch_okutama('Sample.zip')
import cv2, numpy as np, torch
from match import patch_pairs_from_image, train_descriptor
cap = cv2.VideoCapture('/content/data/okutama/1.1.1.mov')
imgs, i = [], 0
while True:
    ok, f = cap.read()
    if not ok or len(imgs) >= 150: break
    if i % 10 == 0:
        s = 1280 / f.shape[1]
        imgs.append(cv2.resize(f, None, fx=s, fy=s))
    i += 1
cap.release()
rng = np.random.default_rng(0)
pa, pb = [], []
for _ in range(6):
    for im in imgs:
        a, b = patch_pairs_from_image(im, rng, n=48)
        if len(a): pa.append(a); pb.append(b)
pa, pb = np.concatenate(pa), np.concatenate(pb)
print(len(pa), 'patch pairs')
net = train_descriptor(pa, pb, epochs=12, device='cuda')
torch.save(net.state_dict(), '/content/tiny_desc.pt')
!cp /content/tiny_desc.pt {OUT}/tiny_desc.pt


In [ ]:
# Evaluate matching accuracy over all 116 sequences for the three methods.
from eval_match import hpatches_sequences, evaluate_pair, aggregate
from eval_restore import write_csv
from match import build_features
def run(kind, weights=None):
    """Evaluate one method over all HPatches sequence pairs."""
    feats = (build_features(kind, device='cuda', weights=weights)
             if kind == 'tinydesc' else build_features(kind))
    rows = []
    for ref, tgt, H, cond, seq, t in hpatches_sequences(config.HPATCHES_DIR,
                                                        resize_width=1024):
        r = evaluate_pair(feats, ref, tgt, H)
        r.update({'method': kind, 'condition': cond})
        rows.append(r)
    return rows
all_rows = run('sift') + run('orb') + run('tinydesc', weights='/content/tiny_desc.pt')
agg = aggregate(all_rows)
for r in agg: print(r)
write_csv(agg, OUT/'matching_hpatches.csv')


In [ ]:
# Evaluate patch matching on the patches release, split into Easy, Hard and Tough.
# The metric is nearest-neighbor matching accuracy, where the correct match
# is the patch with the same index. The official mAP protocol from
# github.com/hpatches/hpatches-benchmark can also be run on the saved
# descriptors.
!curl -sL -o hpp.zip https://huggingface.co/datasets/vbalnt/hpatches/resolve/main/hpatches-release.zip
!unzip -q -o hpp.zip -d /content/data/raw && rm hpp.zip
import cv2, numpy as np, glob, torch
from match import TinyDescNet, describe_patches
net = TinyDescNet(); net.load_state_dict(torch.load('/content/tiny_desc.pt'))
net.eval().cuda()

def patches_of(png):
    """Read one HPatches image and split it into its stacked 65 by 65 patches."""
    im = cv2.imread(png, cv2.IMREAD_GRAYSCALE)
    return im.reshape(-1, 65, 65)

def sift_desc(patches):
    """Compute a SIFT descriptor at the center of each patch."""
    sift = cv2.SIFT_create()
    kp = [cv2.KeyPoint(32.5, 32.5, 52.0)]
    out = []
    for p in patches:
        _, d = sift.compute(p, kp)
        out.append(d[0] if d is not None else np.zeros(128, np.float32))
    return np.array(out, np.float32)

def tiny_desc(patches):
    """Compute TinyDescNet descriptors for each patch, resized to 32 by 32."""
    p32 = np.stack([cv2.resize(p, (32, 32)) for p in patches]).astype(np.float32) / 255.
    return describe_patches(net, p32, device='cuda')

def nn_accuracy(desc_fn, split, max_seqs=60):
    """Return the nearest-neighbor matching accuracy for one split."""
    accs = []
    for seq in sorted(glob.glob('/content/data/raw/hpatches-release/*'))[:max_seqs]:
        d_ref = desc_fn(patches_of(f'{seq}/ref.png'))
        for tgt in sorted(glob.glob(f'{seq}/{split}[0-9]*.png')):
            d_t = desc_fn(patches_of(tgt))
            d2 = ((d_ref**2).sum(1)[:, None] + (d_t**2).sum(1)[None, :]
                  - 2 * d_ref @ d_t.T)
            accs.append(float((d2.argmin(1) == np.arange(len(d_ref))).mean()))
    return float(np.mean(accs))

for split, name in [('e', 'Easy'), ('h', 'Hard'), ('t', 'Tough')]:
    print(f'{name}:  SIFT {nn_accuracy(sift_desc, split):.3f}   '
          f'TinyDesc {nn_accuracy(tiny_desc, split):.3f}')


The sequences table with matching accuracy and homography error is the main
RQ4 result, and the Easy, Hard and Tough splits show where the learned
descriptor performs better. The registration mosaic and the victim map from
`src/register.py` complete the section.
